# Validate Silver project flood mapping

## Purpose

Validate Table 15, `02-silver.silver_project_flood_map`, independently of the transformation. The validator recomputes the project coordinate contract and the MGB source accounting. It also rebuilds every project-to-MGB candidate with its own grid. It checks all seven data-quality attributes with executable checks: Consistency, Accuracy, Completeness, Auditability, Validity, Uniqueness, and Timeliness.

Evidence is persisted to `04-validation.silver_dq_results` **before** the STOP gate runs, so failed blocking evidence stays reviewable. The gate reads the persisted rows. The validator never changes Table 15.

## Cost

The validator parses the selected MGB snapshot twice. One grouped scan builds the source accounting. One spatial join rebuilds the project candidates. The join uses its own bounding-box grid with a different cell size from the transformation. Both sides are filtered before the join: only safe project points and only mapped, usable polygons. All checks come from one metrics row. No cache, Python, UDF, or spatial library is used.

## 1. Declare validation expectations and read upstream context

**What:** declare the expected coordinate screen, the source SRID, the validation grid, and historical reference counts once. Then summarize the published lineage, the newest terminal MGB and DPWH loads, and the latest Bronze, project, and CONFIG validation evidence. Also read the current project run and mapping version.

**Why:** the coordinate screen is an independent expectation here. It is the Table 13 contract from `src/config.py`. The transformation's own constant is not trusted to validate itself. The validator also reads the optional publisher versions straight from `load_log`. A published version must equal that value or stay `NULL`.

**Reference counts are FLAG evidence only:** 63,684 MGB rows and 1,815 blank MGB geometries come from the accepted October 2026 Bronze evidence. They never filter data or decide correctness.

In [0]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `04-validation`;

DECLARE OR REPLACE VARIABLE expected_project_source_system STRING DEFAULT 'DPWH';
DECLARE OR REPLACE VARIABLE mgb_mapping_source_system STRING DEFAULT 'DENR MGB flood susceptibility';
DECLARE OR REPLACE VARIABLE source_geometry_srid INT DEFAULT 4326;
-- table 13 coordinate screen from ph_lat and ph_lon in src/config.py, declared independently here
DECLARE OR REPLACE VARIABLE expected_min_latitude DOUBLE DEFAULT 4.2;
DECLARE OR REPLACE VARIABLE expected_max_latitude DOUBLE DEFAULT 21.3;
DECLARE OR REPLACE VARIABLE expected_min_longitude DOUBLE DEFAULT 116.0;
DECLARE OR REPLACE VARIABLE expected_max_longitude DOUBLE DEFAULT 127.0;
-- a different grid size from the transformation keeps the spatial recomputation independent
DECLARE OR REPLACE VARIABLE validation_grid_cell_degrees DOUBLE DEFAULT 0.25;
DECLARE OR REPLACE VARIABLE reference_mgb_rows BIGINT DEFAULT 63684;
DECLARE OR REPLACE VARIABLE reference_blank_geometry_rows BIGINT DEFAULT 1815;
DECLARE OR REPLACE VARIABLE validation_rule_version STRING
    DEFAULT 'silver-project-flood-map-validation-v1';
DECLARE OR REPLACE VARIABLE expected_rule_version_prefix STRING DEFAULT 'silver-project-flood-map-v2|';
DECLARE OR REPLACE VARIABLE expected_coordinate_screen STRING;

SET VAR expected_coordinate_screen = CONCAT(
    '|coordinate-screen-latitude=', expected_min_latitude, ':', expected_max_latitude,
    '|coordinate-screen-longitude=', expected_min_longitude, ':', expected_max_longitude
);

CREATE OR REPLACE TEMPORARY VIEW project_flood_validation_context AS
WITH published AS (
    SELECT
        COUNT(*) AS output_rows,
        COUNT(DISTINCT run_id) AS run_count,
        COUNT(DISTINCT project_run_id) AS published_project_run_count,
        COUNT(DISTINCT project_source_snapshot_id) AS published_project_snapshot_count,
        COUNT(DISTINCT mgb_source_snapshot_id) AS published_mgb_snapshot_count,
        COUNT(DISTINCT susceptibility_mapping_version) AS published_mapping_version_count,
        COUNT(DISTINCT classification_rule_version) AS published_rule_version_count,
        MAX(run_id) AS run_id,
        MAX(project_run_id) AS project_run_id,
        MAX(project_source_snapshot_id) AS project_source_snapshot_id,
        MAX(project_source_ingest_run_id) AS project_source_ingest_run_id,
        MAX(mgb_source_snapshot_id) AS mgb_source_snapshot_id,
        MAX(mgb_source_ingest_run_id) AS mgb_source_ingest_run_id,
        MAX(susceptibility_mapping_version) AS susceptibility_mapping_version,
        MAX(classification_rule_version) AS classification_rule_version
    FROM `02-silver`.silver_project_flood_map
),
latest_load AS (
    SELECT * EXCEPT (terminal_rank)
    FROM (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY table_name
            ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
        ) AS terminal_rank
        FROM `01-bronze`.load_log
        WHERE table_name IN ('flood_susceptibility', 'dpwh_projects')
          AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
    )
    WHERE terminal_rank = 1
),
load_summary AS (
    SELECT
        COUNT_IF(
            status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')
            AND snapshot_id IS NOT NULL AND TRIM(snapshot_id) <> ''
            AND rows_loaded > 0
        ) AS safe_latest_loads,
        MAX(CASE WHEN table_name = 'flood_susceptibility' THEN snapshot_id END) AS latest_mgb_snapshot_id,
        MAX(CASE WHEN table_name = 'flood_susceptibility' THEN rows_loaded END) AS latest_mgb_rows_loaded,
        MAX(CASE WHEN table_name = 'dpwh_projects' THEN snapshot_id END) AS latest_dpwh_snapshot_id,
        COALESCE(CONCAT_WS('; ', SORT_ARRAY(COLLECT_LIST(CONCAT(table_name, '=', status)))), '')
            AS latest_load_statuses
    FROM latest_load
),
bronze_validation AS (
    SELECT
        COUNT(DISTINCT table_name) AS validated_source_tables,
        COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) AS bronze_blocking_failures
    FROM (
        SELECT dq.*, DENSE_RANK() OVER (
            PARTITION BY dq.table_name ORDER BY dq.run_ts DESC, dq.run_id DESC
        ) AS validation_rank
        FROM `04-validation`.dq_results AS dq
        INNER JOIN latest_load AS latest
            ON dq.table_name = latest.table_name
           AND dq.snapshot_id = latest.snapshot_id
    )
    WHERE validation_rank = 1
),
config_validation AS (
    SELECT
        COUNT(*) AS config_checks,
        COUNT_IF(action = 'stop' AND status = 'FAIL') AS config_blocking_failures
    FROM (
        SELECT *, DENSE_RANK() OVER (ORDER BY run_ts DESC, run_id DESC) AS validation_rank
        FROM `04-validation`.silver_config_dq_results
        WHERE layer = 'silver_config'
    )
    WHERE validation_rank = 1
),
current_project AS (
    SELECT
        COUNT(*) AS project_rows,
        COUNT(DISTINCT run_id) AS current_project_run_count,
        MAX(run_id) AS current_project_run_id,
        MAX(source_snapshot_id) AS current_project_snapshot_id
    FROM `02-silver`.silver_project
),
project_validation AS (
    SELECT
        COUNT(*) AS project_checks,
        COUNT_IF(dq.action = 'stop' AND dq.status = 'FAIL') AS project_blocking_failures
    FROM `04-validation`.silver_dq_results AS dq
    CROSS JOIN current_project
    WHERE dq.table_name IN ('silver_project', 'silver_dpwh_project_component')
      AND dq.pipeline_run_id = current_project.current_project_run_id
),
current_mapping AS (
    SELECT
        COUNT(DISTINCT mapping_version) AS active_mapping_versions,
        MAX(mapping_version) AS current_mapping_version
    FROM `02-silver`.config_mgb_susceptibility_mapping
    WHERE source_system = mgb_mapping_source_system
      AND approval_status = 'APPROVED'
      AND is_active = TRUE
),
audited_versions AS (
    -- optional publisher labels exist only in load_log and are never inferred
    SELECT
        MAX(CASE WHEN log.table_name = 'flood_susceptibility'
            THEN NULLIF(TRIM(log.source_version), '') END) AS audited_mgb_source_version,
        MAX(CASE WHEN log.table_name = 'dpwh_projects'
            THEN NULLIF(TRIM(log.source_version), '') END) AS audited_project_source_version
    FROM `01-bronze`.load_log AS log
    CROSS JOIN published
    WHERE log.status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')
      AND (
          (log.table_name = 'flood_susceptibility' AND log.run_id = published.mgb_source_ingest_run_id)
          OR (log.table_name = 'dpwh_projects' AND log.run_id = published.project_source_ingest_run_id)
      )
)
SELECT *
FROM published
CROSS JOIN load_summary
CROSS JOIN bronze_validation
CROSS JOIN config_validation
CROSS JOIN current_project
CROSS JOIN project_validation
CROSS JOIN current_mapping
CROSS JOIN audited_versions;

SELECT * FROM project_flood_validation_context;

## 2. Recompute the project coordinate contract

**What:** read every canonical project once and assign the expected coordinate status from the independently declared Table 13 screen.

**Why:** Step 5 compares this status with the published status for every project. It also limits the independent spatial recomputation to `VALID_PAIR` coordinates with a stable identity. This proves that unsafe coordinates never reach spatial matching.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW project_flood_validation_points AS
SELECT
    project.project_key,
    project.contract_id,
    project.latitude_parsed,
    project.longitude_parsed,
    CASE
        WHEN project.latitude_parsed IS NULL AND project.longitude_parsed IS NULL THEN 'MISSING_PAIR'
        WHEN project.latitude_parsed IS NULL OR project.longitude_parsed IS NULL THEN 'PARTIAL_PAIR'
        WHEN project.latitude_parsed NOT BETWEEN expected_min_latitude AND expected_max_latitude
          OR project.longitude_parsed NOT BETWEEN expected_min_longitude AND expected_max_longitude
            THEN 'OUT_OF_RANGE'
        ELSE 'VALID_PAIR'
    END AS expected_coordinate_status,
    project.project_key IS NOT NULL
        AND project.contract_id IS NOT NULL
        AND TRIM(project.contract_id) <> '' AS has_valid_identity
FROM `02-silver`.silver_project AS project;

## 3. Recompute MGB source accounting in one scan

**What:** classify every row of the published MGB snapshot with the published mapping version and the governed Table 14 parser. Then group the rows by raw code, mapping status, and geometry status.

**Why:** these buckets must reconcile to the selected row count and the audited `rows_loaded`. Blank and `No rating` stay in their own raw-code buckets. They are never assigned a standard level.

The view `project_flood_validation_mgb` is the classified source. Step 4 reads it again for the spatial recomputation. That second read parses the snapshot once more, which is the accepted cost of an independent check.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW project_flood_validation_mgb AS
WITH approved AS (
    SELECT
        UPPER(TRIM(mapping.raw_susceptibility_code)) AS raw_susceptibility_code,
        mapping.standardized_level,
        mapping.severity_rank
    FROM `02-silver`.config_mgb_susceptibility_mapping AS mapping
    CROSS JOIN project_flood_validation_context AS context
    WHERE mapping.source_system = mgb_mapping_source_system
      AND mapping.approval_status = 'APPROVED'
      AND mapping.is_active = TRUE
      AND mapping.mapping_version = context.susceptibility_mapping_version
),
source_rows AS (
    SELECT
        NULLIF(UPPER(TRIM(mgb.flood_susceptibility_code)), '') AS raw_susceptibility_code,
        NULLIF(TRIM(mgb.geometry_json), '') AS geometry_text
    FROM `01-bronze`.flood_susceptibility AS mgb
    CROSS JOIN project_flood_validation_context AS context
    WHERE mgb._source_snapshot_id = context.mgb_source_snapshot_id
),
-- the governed table 14 parser, kept identical by a regression test
direct_parse AS (
    SELECT
        esri_text.*,
        -- an Esri JSON row is never GeoJSON, so the GeoJSON readers skip it
        CASE
            WHEN esri_text.esri_rings_text IS NULL
            THEN TRY_TO_GEOMETRY(esri_text.geometry_text)
        END AS published_geometry
    FROM (
        SELECT
            source_rows.*,
            -- Esri JSON keeps its coordinates as text under the "rings" key
            NULLIF(REGEXP_EXTRACT(source_rows.geometry_text, '(?s)"rings"\\s*:\\s*(\\[.*\\])', 1), '') AS esri_rings_text
        FROM source_rows
    ) AS esri_text
),
esri_ring_source AS (
    SELECT
        direct_parse.*,
        -- first reading, built as text: every ring is its own polygon.
        -- Text avoids FROM_JSON arrays, which run out of memory on the largest MGB polygons.
        CASE
            WHEN esri_rings_text IS NULL THEN NULL
            WHEN esri_rings_text RLIKE '\\]\\s*\\]\\s*,\\s*\\[\\s*\\['
            THEN TRY_TO_GEOMETRY(CONCAT(
                '{"type":"MultiPolygon","coordinates":[',
                REGEXP_REPLACE(esri_rings_text, '\\]\\s*\\]\\s*,\\s*\\[\\s*\\[', ']]],[[['),
                ']}'
            ))
            ELSE TRY_TO_GEOMETRY(CONCAT('{"type":"Polygon","coordinates":', esri_rings_text, '}'))
        END AS esri_ring_geometry
    FROM direct_parse
),
esri_ring_checked AS (
    SELECT
        esri_ring_source.*,
        esri_ring_geometry IS NOT NULL AND ST_ISVALID(esri_ring_geometry) AS esri_ring_geometry_is_valid
    FROM esri_ring_source
),
esri_ring_arrays AS (
    SELECT
        esri_ring_checked.*,
        -- ring arrays are read only when the first reading is invalid, to find the holes
        CASE
            WHEN esri_rings_text IS NOT NULL AND NOT esri_ring_geometry_is_valid
            THEN FROM_JSON(esri_rings_text, 'ARRAY<ARRAY<ARRAY<DOUBLE>>>')
        END AS esri_rings
    FROM esri_ring_checked
),
esri_ring_parts AS (
    SELECT
        esri_ring_arrays.*,
        -- shoelace signed area per ring: Esri outer rings run clockwise (negative), holes counterclockwise
        TRANSFORM(
            esri_rings,
            ring -> AGGREGATE(
                SEQUENCE(1, SIZE(ring) - 1),
                CAST(0 AS DOUBLE),
                (area, i) -> area + ring[i - 1][0] * ring[i][1] - ring[i][0] * ring[i - 1][1]
            ) / 2
        ) AS esri_ring_signed_areas
    FROM esri_ring_arrays
),
esri_json_parsed AS (
    SELECT
        esri_ring_parts.*,
        CASE
            WHEN esri_ring_parts.esri_rings_text IS NOT NULL
            THEN ST_TRANSFORM(
                ST_SETSRID(
                    CASE
                        -- a valid first reading is kept as it is
                        WHEN esri_ring_geometry_is_valid
                        THEN esri_ring_geometry
                        -- otherwise read the counterclockwise rings as holes in the clockwise rings,
                        -- as the Esri polygon format defines them; this follows the format, it is not a repair
                        WHEN SIZE(FILTER(esri_ring_signed_areas, area -> area < 0)) > 0
                            AND SIZE(FILTER(esri_ring_signed_areas, area -> area > 0)) > 0
                        THEN ST_DIFFERENCE(
                            TRY_TO_GEOMETRY(TO_JSON(NAMED_STRUCT(
                                'type', 'MultiPolygon',
                                'coordinates', TRANSFORM(
                                    FILTER(
                                        ZIP_WITH(esri_rings, esri_ring_signed_areas, (ring, area) -> NAMED_STRUCT('ring', ring, 'area', area)),
                                        part -> part.area < 0
                                    ),
                                    part -> ARRAY(part.ring)
                                )
                            ))),
                            TRY_TO_GEOMETRY(TO_JSON(NAMED_STRUCT(
                                'type', 'MultiPolygon',
                                'coordinates', TRANSFORM(
                                    FILTER(
                                        ZIP_WITH(esri_rings, esri_ring_signed_areas, (ring, area) -> NAMED_STRUCT('ring', ring, 'area', area)),
                                        part -> part.area > 0
                                    ),
                                    part -> ARRAY(part.ring)
                                )
                            )))
                        )
                        ELSE esri_ring_geometry
                    END,
                    3857
                ),
                4326
            )
            ELSE NULL
        END AS esri_geometry
    FROM esri_ring_parts
),
normalized AS (
    SELECT
        esri_json_parsed.*,
        CASE
            WHEN esri_rings_text IS NULL
            THEN TRY_TO_GEOMETRY(REGEXP_REPLACE(
                REGEXP_REPLACE(geometry_text, '\"(\\[[^\"]*\\])\"', CONCAT(CHAR(36), '1')),
                '\"(-?[0-9]+(?:\\.[0-9]+)?(?:[Ee][+-]?[0-9]+)?)\"',
                CONCAT(CHAR(36), '1')
            ))
        END AS normalized_geometry
    FROM esri_json_parsed
),
parsed AS (
    SELECT
        normalized.*,
        COALESCE(
            published_geometry,
            normalized_geometry,
            esri_geometry
        ) AS mgb_geometry
    FROM normalized
)
SELECT
    parsed.raw_susceptibility_code,
    parsed.mgb_geometry,
    approved.standardized_level AS flood_susceptibility_level,
    approved.severity_rank,
    parsed.published_geometry IS NOT NULL AS parsed_as_published,
    parsed.published_geometry IS NULL AND parsed.normalized_geometry IS NOT NULL
        AS parsed_after_normalization,
    parsed.published_geometry IS NULL AND parsed.normalized_geometry IS NULL
        AND parsed.esri_geometry IS NOT NULL AS parsed_as_esri_json,
    CASE
        WHEN approved.raw_susceptibility_code IS NOT NULL THEN 'MAPPED_APPROVED'
        ELSE 'UNMAPPED'
    END AS susceptibility_mapping_status,
    CASE
        WHEN parsed.geometry_text IS NULL THEN 'BLANK_GEOMETRY'
        WHEN parsed.mgb_geometry IS NULL THEN 'UNPARSEABLE_GEOMETRY'
        WHEN ST_ISEMPTY(parsed.mgb_geometry) THEN 'EMPTY_GEOMETRY'
        WHEN REGEXP_REPLACE(UPPER(ST_GEOMETRYTYPE(parsed.mgb_geometry)), '^ST_', '')
            NOT IN ('POLYGON', 'MULTIPOLYGON') THEN 'UNSUPPORTED_GEOMETRY_TYPE'
        WHEN NOT ST_ISVALID(parsed.mgb_geometry) THEN 'INVALID_GEOMETRY'
        ELSE 'USABLE_GEOMETRY'
    END AS source_geometry_status
FROM parsed
LEFT JOIN approved
    ON parsed.raw_susceptibility_code = approved.raw_susceptibility_code;

CREATE OR REPLACE TEMPORARY VIEW project_flood_validation_mgb_buckets AS
SELECT
    COALESCE(raw_susceptibility_code, '<BLANK>') AS raw_susceptibility_code,
    susceptibility_mapping_status,
    source_geometry_status,
    COUNT(*) AS source_rows,
    COUNT_IF(parsed_as_published) AS parsed_as_published_rows,
    COUNT_IF(parsed_after_normalization) AS parsed_after_normalization_rows,
    COUNT_IF(parsed_as_esri_json) AS esri_json_converted_rows,
    COUNT_IF(source_geometry_status = 'USABLE_GEOMETRY'
        AND ST_SRID(mgb_geometry) <> source_geometry_srid) AS unexpected_srid_rows
FROM project_flood_validation_mgb
GROUP BY ALL;

## 4. Recompute project-to-MGB candidates independently

**What:** rebuild every project's candidate polygons from the validator's own safe points and its own mapped, usable polygons. Then aggregate them to project grain.

**Why:** this is the proof behind the accuracy and completeness checks. Step 5 compares the published `matched_polygon_count`, `matched_level_count`, and `candidate_levels` with this recomputation for every project. A missed match, an extra match, a blank or unapproved code, or an unusable geometry would all show up as a difference.

The predicate is the same `ST_INTERSECTS`. The grid uses `validation_grid_cell_degrees`, which differs from the transformation's cell size. A cell-boundary defect in either grid would therefore surface as a mismatch.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW project_flood_validation_candidates AS
WITH usable_polygons AS (
    SELECT
        flood_susceptibility_level,
        mgb_geometry,
        CAST(FLOOR(ST_XMIN(mgb_geometry) / validation_grid_cell_degrees) AS BIGINT) AS x_min_cell,
        CAST(FLOOR(ST_XMAX(mgb_geometry) / validation_grid_cell_degrees) AS BIGINT) AS x_max_cell,
        CAST(FLOOR(ST_YMIN(mgb_geometry) / validation_grid_cell_degrees) AS BIGINT) AS y_min_cell,
        CAST(FLOOR(ST_YMAX(mgb_geometry) / validation_grid_cell_degrees) AS BIGINT) AS y_max_cell
    FROM project_flood_validation_mgb
    WHERE susceptibility_mapping_status = 'MAPPED_APPROVED'
      AND source_geometry_status = 'USABLE_GEOMETRY'
),
polygon_cells AS (
    SELECT
        polygon.flood_susceptibility_level,
        polygon.mgb_geometry,
        INLINE(FLATTEN(TRANSFORM(
            SEQUENCE(polygon.x_min_cell, polygon.x_max_cell),
            x_cell -> TRANSFORM(
                SEQUENCE(polygon.y_min_cell, polygon.y_max_cell),
                y_cell -> NAMED_STRUCT('grid_x', x_cell, 'grid_y', y_cell)
            )
        )))
    FROM usable_polygons AS polygon
),
usable_points AS (
    SELECT
        project_key,
        ST_POINT(longitude_parsed, latitude_parsed, source_geometry_srid) AS project_point,
        CAST(FLOOR(longitude_parsed / validation_grid_cell_degrees) AS BIGINT) AS grid_x,
        CAST(FLOOR(latitude_parsed / validation_grid_cell_degrees) AS BIGINT) AS grid_y
    FROM project_flood_validation_points
    WHERE expected_coordinate_status = 'VALID_PAIR'
      AND has_valid_identity
)
SELECT
    point.project_key,
    COUNT(*) AS recomputed_polygon_count,
    COUNT(DISTINCT polygon.flood_susceptibility_level) AS recomputed_level_count,
    SORT_ARRAY(COLLECT_SET(polygon.flood_susceptibility_level)) AS recomputed_levels
FROM usable_points AS point
INNER JOIN polygon_cells AS polygon
    ON point.grid_x = polygon.grid_x
   AND point.grid_y = polygon.grid_y
   AND ST_INTERSECTS(polygon.mgb_geometry, point.project_point)
GROUP BY point.project_key;

## 5. Calculate one grouped metric row

**What:** compare every published row with the recomputed coordinate status, the recomputed candidates, and the approved levels. Then combine output, project-universe, source-accounting, mapping, and upstream metrics into one row.

**Why:** every check has an explicit denominator. Coverage uses two denominators: all canonical projects, and projects with a usable coordinate. Neither denominator is hardcoded.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW project_flood_validation_metrics AS
WITH context AS (
    SELECT * FROM project_flood_validation_context
),
approved_levels AS (
    SELECT
        mapping.standardized_level AS flood_susceptibility_level,
        mapping.severity_rank
    FROM `02-silver`.config_mgb_susceptibility_mapping AS mapping
    CROSS JOIN context
    WHERE mapping.source_system = mgb_mapping_source_system
      AND mapping.approval_status = 'APPROVED'
      AND mapping.is_active = TRUE
      AND mapping.mapping_version = context.susceptibility_mapping_version
    GROUP BY mapping.standardized_level, mapping.severity_rank
),
approved_level_set AS (
    SELECT COALESCE(COLLECT_SET(flood_susceptibility_level), CAST(ARRAY() AS ARRAY<STRING>))
        AS approved_level_array
    FROM approved_levels
),
audited_publisher_versions AS (
    SELECT audited_mgb_source_version, audited_project_source_version
    FROM context
),
mapping_metrics AS (
    SELECT
        COUNT(*) AS approved_mapping_rows,
        COUNT(*) - COUNT(DISTINCT UPPER(TRIM(mapping.raw_susceptibility_code))) AS duplicate_approved_codes,
        COUNT(DISTINCT mapping.standardized_level) AS approved_level_count,
        COUNT(DISTINCT mapping.severity_rank) AS approved_rank_count,
        COUNT(DISTINCT NAMED_STRUCT('level', mapping.standardized_level, 'rank', mapping.severity_rank))
            AS approved_level_rank_pairs,
        COUNT_IF(UPPER(TRIM(mapping.raw_susceptibility_code)) IN ('', 'NO RATING')) AS approved_unrated_codes
    FROM `02-silver`.config_mgb_susceptibility_mapping AS mapping
    CROSS JOIN context
    WHERE mapping.source_system = mgb_mapping_source_system
      AND mapping.approval_status = 'APPROVED'
      AND mapping.is_active = TRUE
      AND mapping.mapping_version = context.susceptibility_mapping_version
),
source_metrics AS (
    SELECT
        COALESCE(SUM(source_rows), 0) AS mgb_rows,
        COALESCE(SUM(CASE WHEN susceptibility_mapping_status = 'MAPPED_APPROVED'
            AND source_geometry_status = 'USABLE_GEOMETRY' THEN source_rows END), 0) AS mapped_usable_rows,
        COALESCE(SUM(CASE WHEN susceptibility_mapping_status = 'MAPPED_APPROVED'
            AND source_geometry_status <> 'USABLE_GEOMETRY' THEN source_rows END), 0) AS mapped_unusable_rows,
        COALESCE(SUM(CASE WHEN susceptibility_mapping_status = 'UNMAPPED'
            AND source_geometry_status = 'USABLE_GEOMETRY' THEN source_rows END), 0) AS unmapped_usable_rows,
        COALESCE(SUM(CASE WHEN susceptibility_mapping_status = 'UNMAPPED'
            AND source_geometry_status <> 'USABLE_GEOMETRY' THEN source_rows END), 0) AS unmapped_unusable_rows,
        COALESCE(SUM(CASE WHEN raw_susceptibility_code = '<BLANK>' THEN source_rows END), 0) AS blank_code_rows,
        COALESCE(SUM(CASE WHEN raw_susceptibility_code = 'NO RATING' THEN source_rows END), 0) AS no_rating_rows,
        COALESCE(SUM(CASE WHEN source_geometry_status = 'USABLE_GEOMETRY' THEN source_rows END), 0) AS usable_geometry_rows,
        COALESCE(SUM(CASE WHEN source_geometry_status = 'BLANK_GEOMETRY' THEN source_rows END), 0) AS blank_geometry_rows,
        COALESCE(SUM(CASE WHEN source_geometry_status = 'UNPARSEABLE_GEOMETRY' THEN source_rows END), 0)
            AS unparseable_geometry_rows,
        COALESCE(SUM(CASE WHEN source_geometry_status IN ('EMPTY_GEOMETRY', 'UNSUPPORTED_GEOMETRY_TYPE')
            THEN source_rows END), 0) AS empty_or_unsupported_geometry_rows,
        COALESCE(SUM(CASE WHEN source_geometry_status = 'INVALID_GEOMETRY' THEN source_rows END), 0)
            AS invalid_geometry_rows,
        COALESCE(SUM(parsed_as_published_rows), 0) AS parsed_as_published_rows,
        COALESCE(SUM(parsed_after_normalization_rows), 0) AS parsed_after_normalization_rows,
        COALESCE(SUM(esri_json_converted_rows), 0) AS esri_json_converted_rows,
        COALESCE(SUM(unexpected_srid_rows), 0) AS unexpected_srid_rows,
        COALESCE(CONCAT_WS('; ', SORT_ARRAY(COLLECT_LIST(
            CONCAT(raw_susceptibility_code, '/', susceptibility_mapping_status, '/',
                   source_geometry_status, '=', source_rows)
        ))), '') AS source_bucket_details
    FROM project_flood_validation_mgb_buckets
),
row_evidence AS (
    SELECT
        published.*,
        point.project_key IS NOT NULL AS has_canonical_project,
        point.expected_coordinate_status,
        point.latitude_parsed AS source_latitude,
        point.longitude_parsed AS source_longitude,
        COALESCE(recomputed.recomputed_polygon_count, 0) AS recomputed_polygon_count,
        COALESCE(recomputed.recomputed_level_count, 0) AS recomputed_level_count,
        COALESCE(recomputed.recomputed_levels, CAST(ARRAY() AS ARRAY<STRING>)) AS recomputed_levels,
        level.flood_susceptibility_level IS NOT NULL AS is_approved_level,
        level.severity_rank AS approved_severity_rank,
        level_set.approved_level_array,
        audited.audited_mgb_source_version,
        audited.audited_project_source_version
    FROM `02-silver`.silver_project_flood_map AS published
    CROSS JOIN approved_level_set AS level_set
    CROSS JOIN audited_publisher_versions AS audited
    LEFT JOIN project_flood_validation_points AS point
        ON published.project_key = point.project_key
    LEFT JOIN project_flood_validation_candidates AS recomputed
        ON published.project_key = recomputed.project_key
    LEFT JOIN approved_levels AS level
        ON published.flood_susceptibility_level = level.flood_susceptibility_level
),
output_metrics AS (
    SELECT
        COUNT(*) - COUNT(DISTINCT project_flood_map_key) AS duplicate_keys,
        COUNT_IF(contract_id IS NOT NULL AND TRIM(contract_id) <> '')
            - COUNT(DISTINCT CASE WHEN contract_id IS NOT NULL AND TRIM(contract_id) <> ''
                THEN NAMED_STRUCT('source_system', source_system, 'contract_id', contract_id, 'run_id', run_id)
            END) AS duplicate_grain_rows,
        COUNT(*) - COUNT(DISTINCT project_key) AS duplicate_project_rows,
        COUNT_IF(project_flood_map_key <> SHA2(CONCAT_WS('|', source_system, project_key, run_id), 256))
            AS invalid_keys,
        COUNT_IF(run_id <> SHA2(CONCAT_WS(
            '||',
            project_run_id, project_source_snapshot_id, project_source_ingest_run_id,
            mgb_source_snapshot_id, mgb_source_ingest_run_id,
            COALESCE(mgb_source_version, 'NO_PUBLISHER_VERSION'),
            susceptibility_mapping_version, mgb_mapping_source_version,
            classification_rule, classification_rule_version
        ), 256)) AS invalid_run_ids,
        COUNT_IF(match_status = 'MATCHED') AS matched_rows,
        COUNT_IF(match_status = 'AMBIGUOUS') AS ambiguous_rows,
        COUNT_IF(match_status = 'UNMATCHED') AS unmatched_rows,
        COUNT_IF(match_status = 'INVALID_SOURCE') AS invalid_source_rows,
        COUNT_IF(coordinate_status = 'VALID_PAIR') AS usable_coordinate_rows,
        COUNT_IF(coordinate_status = 'MISSING_PAIR') AS missing_coordinate_rows,
        COUNT_IF(coordinate_status = 'PARTIAL_PAIR') AS partial_coordinate_rows,
        COUNT_IF(coordinate_status = 'OUT_OF_RANGE') AS out_of_range_coordinate_rows,
        COUNT_IF(coordinate_status = 'VALID_PAIR' AND match_status = 'MATCHED') AS usable_matched_rows,
        COUNT_IF(coordinate_status = 'VALID_PAIR' AND match_status = 'AMBIGUOUS') AS usable_ambiguous_rows,
        COUNT_IF(exception_reason = 'NO_APPROVED_MGB_INTERSECTION') AS no_intersection_rows,
        COUNT_IF(match_status = 'MATCHED' AND matched_polygon_count > 1) AS same_level_multiple_polygon_rows,
        COUNT_IF(
            match_status IS NULL OR match_status NOT IN ('MATCHED', 'AMBIGUOUS', 'UNMATCHED', 'INVALID_SOURCE')
            OR match_method IS NULL OR match_method NOT IN ('COORDINATE_MGB_POLYGON', 'NONE')
            OR match_quality IS NULL OR match_quality NOT IN ('SPATIAL', 'REVIEW_REQUIRED', 'UNRESOLVED')
            OR coordinate_status IS NULL
            OR coordinate_status NOT IN ('VALID_PAIR', 'MISSING_PAIR', 'PARTIAL_PAIR', 'OUT_OF_RANGE')
            OR (ambiguity_reason IS NOT NULL AND ambiguity_reason <> 'MULTIPLE_APPROVED_FLOOD_LEVELS')
            OR (exception_reason IS NOT NULL AND exception_reason NOT IN (
                'MISSING_COORDINATE_PAIR', 'PARTIAL_COORDINATE_PAIR', 'COORDINATE_OUT_OF_RANGE',
                'NO_APPROVED_MGB_INTERSECTION', 'MULTIPLE_APPROVED_FLOOD_LEVELS',
                'MISSING_STABLE_SOURCE_IDENTITY'
            ))
        ) AS invalid_status_values,
        COUNT_IF(
            matched_polygon_count IS NULL OR matched_polygon_count < 0
            OR matched_level_count IS NULL OR matched_level_count < 0
            OR matched_polygon_count < matched_level_count
            OR candidate_levels IS NULL OR SIZE(candidate_levels) <> matched_level_count
        ) AS invalid_count_rows,
        COUNT_IF(match_status = 'MATCHED' AND (
            matched_level_count <> 1 OR matched_polygon_count < 1
            OR flood_susceptibility_level IS NULL OR severity_rank IS NULL
            OR flood_susceptibility_level IS DISTINCT FROM TRY_ELEMENT_AT(candidate_levels, 1)
            OR match_method <> 'COORDINATE_MGB_POLYGON' OR match_quality <> 'SPATIAL'
            OR ambiguity_reason IS NOT NULL OR exception_reason IS NOT NULL
        )) AS invalid_matched_rows,
        COUNT_IF(match_status = 'AMBIGUOUS' AND (
            matched_level_count <= 1
            OR flood_susceptibility_level IS NOT NULL OR severity_rank IS NOT NULL
            OR match_method <> 'COORDINATE_MGB_POLYGON' OR match_quality <> 'REVIEW_REQUIRED'
            OR ambiguity_reason IS DISTINCT FROM 'MULTIPLE_APPROVED_FLOOD_LEVELS'
            OR exception_reason IS DISTINCT FROM 'MULTIPLE_APPROVED_FLOOD_LEVELS'
        )) AS invalid_ambiguous_rows,
        COUNT_IF(match_status IN ('UNMATCHED', 'INVALID_SOURCE') AND (
            matched_polygon_count <> 0 OR matched_level_count <> 0
            OR flood_susceptibility_level IS NOT NULL OR severity_rank IS NOT NULL
            OR match_method <> 'NONE' OR match_quality <> 'UNRESOLVED'
            OR ambiguity_reason IS NOT NULL OR exception_reason IS NULL
        )) AS invalid_unresolved_rows,
        COUNT_IF(
            (match_status = 'INVALID_SOURCE'
                AND exception_reason IS DISTINCT FROM 'MISSING_STABLE_SOURCE_IDENTITY')
            OR (match_status = 'UNMATCHED' AND exception_reason IS DISTINCT FROM CASE coordinate_status
                WHEN 'MISSING_PAIR' THEN 'MISSING_COORDINATE_PAIR'
                WHEN 'PARTIAL_PAIR' THEN 'PARTIAL_COORDINATE_PAIR'
                WHEN 'OUT_OF_RANGE' THEN 'COORDINATE_OUT_OF_RANGE'
                WHEN 'VALID_PAIR' THEN 'NO_APPROVED_MGB_INTERSECTION'
            END)
        ) AS invalid_reason_rows,
        COUNT_IF(match_status = 'MATCHED'
            AND (NOT is_approved_level OR severity_rank IS DISTINCT FROM approved_severity_rank))
            AS level_rank_contract_mismatches,
        COUNT_IF(SIZE(ARRAY_EXCEPT(candidate_levels, approved_level_array)) > 0)
            AS unapproved_candidate_level_rows,
        COUNT_IF(
            matched_polygon_count <> recomputed_polygon_count
            OR matched_level_count <> recomputed_level_count
            OR NOT (candidate_levels <=> recomputed_levels)
        ) AS spatial_evidence_mismatches,
        COUNT_IF(match_status = 'MATCHED' AND recomputed_polygon_count = 0) AS matched_without_intersection_rows,
        COUNT_IF(recomputed_level_count > 1
            AND (match_status <> 'AMBIGUOUS' OR flood_susceptibility_level IS NOT NULL))
            AS collapsed_multiple_level_rows,
        COUNT_IF(recomputed_level_count = 1 AND recomputed_polygon_count > 1
            AND (match_status <> 'MATCHED' OR matched_polygon_count <> recomputed_polygon_count))
            AS unresolved_same_level_overlap_rows,
        COUNT_IF(
            NOT has_canonical_project
            OR coordinate_status IS DISTINCT FROM expected_coordinate_status
            OR latitude IS DISTINCT FROM source_latitude
            OR longitude IS DISTINCT FROM source_longitude
        ) AS coordinate_contract_mismatches,
        COUNT_IF(matched_polygon_count > 0 AND expected_coordinate_status IS DISTINCT FROM 'VALID_PAIR')
            AS unsafe_coordinate_match_rows,
        COUNT_IF(
            project_flood_map_key IS NULL OR source_system IS NULL OR project_key IS NULL
            OR classification_rule IS NULL OR classification_rule_version IS NULL
            OR project_source_snapshot_id IS NULL OR project_source_ingest_run_id IS NULL
            OR project_run_id IS NULL OR project_rule_version IS NULL OR project_source_load_ts IS NULL
            OR mgb_source_snapshot_id IS NULL OR mgb_source_ingest_run_id IS NULL OR mgb_source_load_ts IS NULL
            OR susceptibility_mapping_version IS NULL OR mgb_mapping_source_version IS NULL
            OR run_id IS NULL OR source_load_ts IS NULL
        ) AS missing_mandatory_lineage_rows,
        COUNT_IF(source_system IS DISTINCT FROM expected_project_source_system) AS unexpected_source_system_rows,
        COUNT_IF(
            NOT STARTSWITH(classification_rule_version, expected_rule_version_prefix)
            OR INSTR(classification_rule_version, expected_coordinate_screen) = 0
        ) AS rule_version_contract_mismatches,
        COUNT_IF(mgb_source_version IS NULL) AS missing_mgb_publisher_version_rows,
        COUNT_IF(project_source_version IS NULL) AS missing_project_publisher_version_rows,
        COUNT_IF(
            NOT (mgb_source_version <=> audited_mgb_source_version)
            OR NOT (project_source_version <=> audited_project_source_version)
        ) AS unaudited_publisher_version_rows
    FROM row_evidence
),
universe_metrics AS (
    SELECT
        (SELECT COUNT(*) FROM `02-silver`.silver_project AS project
         LEFT ANTI JOIN `02-silver`.silver_project_flood_map AS published
             ON project.project_key = published.project_key) AS missing_project_rows,
        (SELECT COUNT(*) FROM `02-silver`.silver_project_flood_map AS published
         LEFT ANTI JOIN `02-silver`.silver_project AS project
             ON published.project_key = project.project_key) AS extra_project_rows
)
SELECT *
FROM context
CROSS JOIN mapping_metrics
CROSS JOIN source_metrics
CROSS JOIN output_metrics
CROSS JOIN universe_metrics;

## 6. Express STOP and FLAG checks across all seven attributes

**STOP** protects correctness:

- safe and current upstream loads and validation
- MGB source accounting and approved-only mapping
- every canonical project present exactly once
- unique keys and grain, and reproducible keys and run ID
- controlled values and status combinations that agree with candidate counts
- one distinct level for `MATCHED`, no level for `AMBIGUOUS` or `UNMATCHED`, and no severity collapse
- published candidates that equal the independent recomputation
- the Table 13 coordinate contract and mandatory lineage

**FLAG** keeps real limitations visible with denominators: projects without usable coordinates, usable points without an approved polygon, ambiguous points, and same-level overlap. It also covers unmapped or unusable MGB rows, optional publisher-version gaps, and differences from historical references. A FLAG is not a pipeline failure.

A failed blocking check is `FAIL`. A nonzero review finding is `FLAG`. Zero findings are `PASS`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW project_flood_validation_checks AS
WITH checks AS (
    -- one metrics row is unpivoted into check rows, so no source scan repeats per check
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Latest MGB and DPWH loads are successful and identified',
            'failed_rows', CAST(2 - safe_latest_loads AS BIGINT), 'total_rows', CAST(2 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Latest terminal loads: ', latest_load_statuses)
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Latest MGB and DPWH Bronze validation has no blocking failure',
            'failed_rows', CAST(bronze_blocking_failures + (2 - validated_source_tables) AS BIGINT), 'total_rows', CAST(2 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Validated source tables=', validated_source_tables, '; blocking failures=', bronze_blocking_failures)
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Published MGB snapshot is the latest validated MGB snapshot',
            'failed_rows', CAST(CASE WHEN mgb_source_snapshot_id <=> latest_mgb_snapshot_id THEN 0 ELSE 1 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Published=', COALESCE(mgb_source_snapshot_id, 'NULL'), '; latest=', COALESCE(latest_mgb_snapshot_id, 'NULL'))
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Published project snapshot is the latest validated DPWH snapshot',
            'failed_rows', CAST(CASE WHEN project_source_snapshot_id <=> latest_dpwh_snapshot_id THEN 0 ELSE 1 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Published=', COALESCE(project_source_snapshot_id, 'NULL'), '; latest=', COALESCE(latest_dpwh_snapshot_id, 'NULL'))
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Published project run is the current accepted Silver project run',
            'failed_rows', CAST(CASE WHEN current_project_run_count = 1 AND project_run_id <=> current_project_run_id
                AND project_source_snapshot_id <=> current_project_snapshot_id THEN 0 ELSE 1 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Published run=', COALESCE(project_run_id, 'NULL'), '; current run=', COALESCE(current_project_run_id, 'NULL'), '; current runs=', current_project_run_count)
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Selected Silver project validation is present and has no blocking failure',
            'failed_rows', CAST(project_blocking_failures + CASE WHEN project_checks = 0 THEN 1 ELSE 0 END AS BIGINT), 'total_rows', CAST(GREATEST(project_checks, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Selected project checks=', project_checks, '; blocking failures=', project_blocking_failures)
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Published mapping version is the single current approved version',
            'failed_rows', CAST(CASE WHEN active_mapping_versions = 1 AND susceptibility_mapping_version <=> current_mapping_version THEN 0 ELSE 1 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Published=', COALESCE(susceptibility_mapping_version, 'NULL'), '; current=', COALESCE(current_mapping_version, 'NULL'), '; active approved versions=', active_mapping_versions)
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'One deterministic run represents the table',
            'failed_rows', CAST(ABS(run_count - 1) AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Distinct run IDs=', run_count)
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Runtime MGB counts are compared with historical Bronze references',
            'failed_rows', CAST(ABS(mgb_rows - reference_mgb_rows) + ABS(blank_geometry_rows - reference_blank_geometry_rows) AS BIGINT), 'total_rows', CAST(mgb_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Runtime rows=', mgb_rows, ' versus reference ', reference_mgb_rows, '; runtime blank geometry=', blank_geometry_rows, ' versus reference ', reference_blank_geometry_rows, '; references are evidence, not filters')
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Silver CONFIG validation is present and has no blocking failure',
            'failed_rows', CAST(config_blocking_failures + CASE WHEN config_checks = 0 THEN 1 ELSE 0 END AS BIGINT), 'total_rows', CAST(GREATEST(config_checks, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Latest CONFIG checks=', config_checks, '; blocking failures=', config_blocking_failures)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'One project run, MGB snapshot, mapping version, and rule version are published',
            'failed_rows', CAST(ABS(published_project_run_count - 1) + ABS(published_project_snapshot_count - 1) + ABS(published_mgb_snapshot_count - 1) + ABS(published_mapping_version_count - 1) + ABS(published_rule_version_count - 1) AS BIGINT), 'total_rows', CAST(5 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Project runs=', published_project_run_count, '; project snapshots=', published_project_snapshot_count, '; MGB snapshots=', published_mgb_snapshot_count, '; mapping versions=', published_mapping_version_count, '; rule versions=', published_rule_version_count)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Approved MGB mapping has one target per raw code and one-to-one levels and ranks',
            'failed_rows', CAST(duplicate_approved_codes + ABS(approved_level_count - approved_rank_count) + ABS(approved_level_count - approved_level_rank_pairs) + CASE WHEN approved_mapping_rows = 0 THEN 1 ELSE 0 END AS BIGINT), 'total_rows', CAST(GREATEST(approved_mapping_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Approved rows=', approved_mapping_rows, '; levels=', approved_level_count, '; ranks=', approved_rank_count, '; duplicate codes=', duplicate_approved_codes)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Published levels and ranks match the approved configuration',
            'failed_rows', CAST(level_rank_contract_mismatches AS BIGINT), 'total_rows', CAST(GREATEST(matched_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Matched rows whose level is not approved or whose rank differs from configuration=', level_rank_contract_mismatches)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Final statuses account for every output row',
            'failed_rows', CAST(ABS(output_rows - matched_rows - ambiguous_rows - unmatched_rows - invalid_source_rows) AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Matched=', matched_rows, '; ambiguous=', ambiguous_rows, '; unmatched=', unmatched_rows, '; invalid source=', invalid_source_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Ambiguous rows publish multiple candidate levels and no final level',
            'failed_rows', CAST(invalid_ambiguous_rows AS BIGINT), 'total_rows', CAST(GREATEST(ambiguous_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Ambiguous rows with one level, a published level or rank, or a wrong method, quality, or reason=', invalid_ambiguous_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Unmatched and invalid rows publish no level, rank, or candidate',
            'failed_rows', CAST(invalid_unresolved_rows AS BIGINT), 'total_rows', CAST(GREATEST(unmatched_rows + invalid_source_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Unresolved rows carrying a level, rank, candidate, wrong method or quality, or no reason=', invalid_unresolved_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Published coordinate screen equals the Table 13 contract',
            'failed_rows', CAST(rule_version_contract_mismatches AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Published rule version=', COALESCE(classification_rule_version, 'NULL'), '; expected screen=', expected_coordinate_screen)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Published candidate evidence equals an independent spatial recomputation',
            'failed_rows', CAST(spatial_evidence_mismatches AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows whose polygon count, level count, or candidate levels differ from the recomputation=', spatial_evidence_mismatches, '; validation grid degrees=', validation_grid_cell_degrees)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Matched project points intersect at least one approved usable polygon',
            'failed_rows', CAST(matched_without_intersection_rows AS BIGINT), 'total_rows', CAST(GREATEST(matched_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Matched rows without a recomputed intersection=', matched_without_intersection_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Multiple distinct levels are never collapsed by severity',
            'failed_rows', CAST(collapsed_multiple_level_rows AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows with more than one recomputed level that are not ambiguous or publish a level=', collapsed_multiple_level_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Same-level overlapping polygons resolve to one level and keep their count',
            'failed_rows', CAST(unresolved_same_level_overlap_rows AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows with several same-level polygons that are not matched or lost their polygon count=', unresolved_same_level_overlap_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Blank, No rating, and unapproved codes never drive a classification',
            'failed_rows', CAST(unapproved_candidate_level_rows AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows with a candidate level outside the approved configuration=', unapproved_candidate_level_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Usable MGB geometry uses the expected longitude-latitude SRID',
            'failed_rows', CAST(unexpected_srid_rows AS BIGINT), 'total_rows', CAST(GREATEST(usable_geometry_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Usable rows with an unexpected SRID=', unexpected_srid_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Approval of blank or No rating codes remains visible',
            'failed_rows', CAST(approved_unrated_codes AS BIGINT), 'total_rows', CAST(GREATEST(approved_mapping_rows, 1) AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Approved mapping rows for blank or No rating=', approved_unrated_codes, '; zero means these values stay unresolved')
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Every canonical project has exactly one output row',
            'failed_rows', CAST(missing_project_rows + extra_project_rows + ABS(output_rows - project_rows) AS BIGINT), 'total_rows', CAST(GREATEST(project_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Canonical projects=', project_rows, '; output rows=', output_rows, '; missing=', missing_project_rows, '; extra=', extra_project_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'MGB source rows reconcile to mapping and geometry buckets',
            'failed_rows', CAST(ABS(mgb_rows - mapped_usable_rows - mapped_unusable_rows - unmapped_usable_rows - unmapped_unusable_rows) + ABS(mgb_rows - COALESCE(latest_mgb_rows_loaded, 0)) AS BIGINT), 'total_rows', CAST(GREATEST(mgb_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Selected rows=', mgb_rows, '; audited rows=', COALESCE(latest_mgb_rows_loaded, 0), '; mapped usable=', mapped_usable_rows, '; mapped unusable=', mapped_unusable_rows, '; unmapped usable=', unmapped_usable_rows, '; unmapped unusable=', unmapped_unusable_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Flood classification coverage is reported against all projects',
            'failed_rows', CAST(output_rows - matched_rows AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Denominator=', output_rows, '; matched=', matched_rows, '; matched percent=', COALESCE(CAST(ROUND(100.0 * matched_rows / NULLIF(output_rows, 0), 2) AS STRING), 'NULL'), '; ambiguous=', ambiguous_rows, '; unmatched=', unmatched_rows, '; invalid source=', invalid_source_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Flood classification coverage is reported against usable coordinates',
            'failed_rows', CAST(usable_coordinate_rows - usable_matched_rows AS BIGINT), 'total_rows', CAST(GREATEST(usable_coordinate_rows, 1) AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Usable-coordinate denominator=', usable_coordinate_rows, '; matched=', usable_matched_rows, '; matched percent=', COALESCE(CAST(ROUND(100.0 * usable_matched_rows / NULLIF(usable_coordinate_rows, 0), 2) AS STRING), 'NULL'), '; ambiguous=', usable_ambiguous_rows, '; no approved intersection=', no_intersection_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Projects without usable coordinates remain present',
            'failed_rows', CAST(output_rows - usable_coordinate_rows AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Missing pair=', missing_coordinate_rows, '; partial pair=', partial_coordinate_rows, '; out of range=', out_of_range_coordinate_rows, '; denominator=', output_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Usable points without an approved MGB intersection remain visible',
            'failed_rows', CAST(no_intersection_rows AS BIGINT), 'total_rows', CAST(GREATEST(usable_coordinate_rows, 1) AS BIGINT),
            'action', 'flag',
            'details', CONCAT('No approved intersection=', no_intersection_rows, '; usable-coordinate denominator=', usable_coordinate_rows, '; this is not low or zero risk')
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Unmapped or unrated MGB rows remain visible',
            'failed_rows', CAST(unmapped_usable_rows + unmapped_unusable_rows AS BIGINT), 'total_rows', CAST(GREATEST(mgb_rows, 1) AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Blank code=', blank_code_rows, '; No rating=', no_rating_rows, '; other unmapped=', unmapped_usable_rows + unmapped_unusable_rows - blank_code_rows - no_rating_rows, '; buckets: ', source_bucket_details)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Mapped MGB rows without usable geometry remain visible',
            'failed_rows', CAST(mapped_unusable_rows AS BIGINT), 'total_rows', CAST(GREATEST(mapped_usable_rows + mapped_unusable_rows, 1) AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Mapped rows excluded from matching=', mapped_unusable_rows, '; mapped usable=', mapped_usable_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Auditability', 'check_name', 'Mandatory project, MGB, mapping, rule, and run lineage is present',
            'failed_rows', CAST(missing_mandatory_lineage_rows + unexpected_source_system_rows AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows missing mandatory lineage=', missing_mandatory_lineage_rows, '; rows with an unexpected source system=', unexpected_source_system_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Auditability', 'check_name', 'Project flood map keys reproduce from source system, project, and run',
            'failed_rows', CAST(invalid_keys AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Invalid deterministic keys=', invalid_keys)
        ),
        NAMED_STRUCT(
            'attribute', 'Auditability', 'check_name', 'Run ID reproduces from published project, MGB, mapping, and rule lineage',
            'failed_rows', CAST(invalid_run_ids AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows whose run_id does not reproduce=', invalid_run_ids)
        ),
        NAMED_STRUCT(
            'attribute', 'Auditability', 'check_name', 'Publisher versions come only from load_log and are never fabricated',
            'failed_rows', CAST(unaudited_publisher_version_rows AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows whose MGB or DPWH publisher version differs from load_log=', unaudited_publisher_version_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Auditability', 'check_name', 'Optional MGB publisher version gaps remain visible',
            'failed_rows', CAST(missing_mgb_publisher_version_rows AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Rows without an MGB publisher version=', missing_mgb_publisher_version_rows, '; snapshot lineage remains mandatory')
        ),
        NAMED_STRUCT(
            'attribute', 'Auditability', 'check_name', 'Optional DPWH publisher version gaps remain visible',
            'failed_rows', CAST(missing_project_publisher_version_rows AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Rows without a DPWH publisher version=', missing_project_publisher_version_rows, '; snapshot lineage remains mandatory')
        ),
        NAMED_STRUCT(
            'attribute', 'Auditability', 'check_name', 'MGB geometry parse paths are reported',
            'failed_rows', CAST(0 AS BIGINT), 'total_rows', CAST(GREATEST(mgb_rows, 1) AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Parsed as published=', parsed_as_published_rows, '; parsed after quoted-number normalization=', parsed_after_normalization_rows, '; parsed from Esri rings=', esri_json_converted_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Statuses, methods, qualities, and reasons use the controlled vocabulary',
            'failed_rows', CAST(invalid_status_values AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows with uncontrolled values=', invalid_status_values)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Candidate counts are non-negative and agree with candidate levels',
            'failed_rows', CAST(invalid_count_rows AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows with null, negative, or inconsistent candidate counts=', invalid_count_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Matched rows publish exactly one distinct candidate level',
            'failed_rows', CAST(invalid_matched_rows AS BIGINT), 'total_rows', CAST(GREATEST(matched_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Matched rows without exactly one level, a polygon, a level and rank, or the expected method and quality=', invalid_matched_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Unresolved reasons follow the coordinate contract',
            'failed_rows', CAST(invalid_reason_rows AS BIGINT), 'total_rows', CAST(GREATEST(unmatched_rows + invalid_source_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Unmatched or invalid rows with a reason that does not match their coordinate status=', invalid_reason_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Published coordinates and statuses follow the Table 13 contract',
            'failed_rows', CAST(coordinate_contract_mismatches AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows whose coordinates or coordinate status differ from the recomputed contract=', coordinate_contract_mismatches)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Only safe coordinates take part in spatial matching',
            'failed_rows', CAST(unsafe_coordinate_match_rows AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows with candidates but no VALID_PAIR coordinate=', unsafe_coordinate_match_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Blank MGB geometry remains visible',
            'failed_rows', CAST(blank_geometry_rows AS BIGINT), 'total_rows', CAST(GREATEST(mgb_rows, 1) AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Blank geometry=', blank_geometry_rows, '; historical reference=', reference_blank_geometry_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Unparseable, empty, or unsupported MGB geometry remains visible',
            'failed_rows', CAST(unparseable_geometry_rows + empty_or_unsupported_geometry_rows AS BIGINT), 'total_rows', CAST(GREATEST(mgb_rows, 1) AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Unparseable=', unparseable_geometry_rows, '; empty or non-polygon=', empty_or_unsupported_geometry_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Invalid MGB geometry is excluded without repair',
            'failed_rows', CAST(invalid_geometry_rows AS BIGINT), 'total_rows', CAST(GREATEST(mgb_rows, 1) AS BIGINT),
            'action', 'flag',
            'details', CONCAT('OGC-invalid geometry excluded from matching=', invalid_geometry_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Projects across multiple approved levels remain ambiguous and visible',
            'failed_rows', CAST(ambiguous_rows AS BIGINT), 'total_rows', CAST(GREATEST(usable_coordinate_rows, 1) AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Ambiguous projects=', ambiguous_rows, '; usable-coordinate denominator=', usable_coordinate_rows, '; no severity precedence is applied')
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Projects inside several same-level polygons remain visible',
            'failed_rows', CAST(same_level_multiple_polygon_rows AS BIGINT), 'total_rows', CAST(GREATEST(matched_rows, 1) AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Matched projects with more than one same-level polygon=', same_level_multiple_polygon_rows, '; matched denominator=', matched_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Uniqueness', 'check_name', 'Project flood map keys are unique',
            'failed_rows', CAST(duplicate_keys AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Duplicate keys=', duplicate_keys)
        ),
        NAMED_STRUCT(
            'attribute', 'Uniqueness', 'check_name', 'One row exists per source system, Contract ID, and run',
            'failed_rows', CAST(duplicate_grain_rows AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Duplicate usable source-system, Contract-ID, run rows=', duplicate_grain_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Uniqueness', 'check_name', 'No project fans out into more than one row',
            'failed_rows', CAST(duplicate_project_rows AS BIGINT), 'total_rows', CAST(GREATEST(output_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Duplicate project keys=', duplicate_project_rows)
        )
    ))
    FROM project_flood_validation_metrics
),
context AS (
    SELECT
        run_id AS pipeline_run_id,
        project_source_snapshot_id AS source_snapshot_id,
        classification_rule_version AS rule_version,
        SHA2(CONCAT_WS('|', run_id, validation_rule_version), 256) AS validation_run_id,
        CURRENT_TIMESTAMP() AS run_ts
    FROM project_flood_validation_context
)
SELECT
    context.validation_run_id, context.pipeline_run_id, 'silver' AS layer,
    'silver_project_flood_map' AS table_name, context.source_snapshot_id, context.rule_version,
    checks.attribute AS data_quality_attribute, checks.check_name AS data_quality_check,
    checks.failed_rows, checks.total_rows,
    CASE WHEN checks.total_rows > 0
        THEN ROUND(100.0 * checks.failed_rows / checks.total_rows, 4) END AS percentage,
    CASE
        WHEN checks.failed_rows = 0 THEN 'PASS'
        WHEN checks.action = 'stop' THEN 'FAIL'
        ELSE 'FLAG'
    END AS status,
    checks.action, checks.details, context.run_ts
FROM checks
CROSS JOIN context;

## 7. Persist evidence before enforcing the gate

**What:** `MERGE` the checks into the shared `04-validation.silver_dq_results` table. Then enforce the STOP gate from the persisted rows.

**Why:** the validation run ID is `SHA2(run_id | validation_rule_version)`. An exact rerun therefore updates the same evidence instead of appending duplicates. The gate reads stored evidence, so the expensive checks run once and failed STOP evidence stays reviewable.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS `04-validation`.silver_dq_results (
    validation_run_id STRING NOT NULL, pipeline_run_id STRING NOT NULL, layer STRING NOT NULL,
    table_name STRING NOT NULL, source_snapshot_id STRING, rule_version STRING,
    data_quality_attribute STRING NOT NULL, data_quality_check STRING NOT NULL,
    failed_rows BIGINT, total_rows BIGINT, percentage DOUBLE, status STRING NOT NULL,
    action STRING NOT NULL, details STRING, run_ts TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Reusable Silver transformation validation evidence';

MERGE INTO `04-validation`.silver_dq_results AS target
USING project_flood_validation_checks AS source
ON target.validation_run_id = source.validation_run_id
AND target.table_name = source.table_name
AND target.data_quality_check = source.data_quality_check
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

DECLARE OR REPLACE VARIABLE project_flood_validation_run_id STRING;
SET VAR project_flood_validation_run_id = (
    SELECT SHA2(CONCAT_WS('|', MAX(run_id), validation_rule_version), 256)
    FROM `02-silver`.silver_project_flood_map
);

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(status = 'FAIL' AND action = 'stop') = 0,
    CONCAT(
        COUNT_IF(status = 'FAIL' AND action = 'stop'),
        ' blocking Silver project flood-mapping checks failed or no evidence was persisted'
    )
)
FROM `04-validation`.silver_dq_results
WHERE table_name = 'silver_project_flood_map'
  AND validation_run_id = project_flood_validation_run_id;

## Summary

Validation protects the exact Table 15 grain: one canonical project per deterministic project-flood mapping run. Every canonical project must appear exactly once. Keys and the run ID must reproduce. Statuses must agree with candidate counts, levels, ranks, and reasons.

The strongest checks are independent. The validator recomputes the coordinate contract from its own screen and reparses MGB with the governed parser. It then rebuilds every project's candidates with a different grid. Published candidate evidence must equal that recomputation. Multiple distinct levels must stay ambiguous, and same-level overlap must stay matched.

Projects without usable coordinates and usable points without an approved polygon remain visible as FLAG findings with denominators. So do ambiguous projects, unmapped or unusable MGB rows, optional version gaps, and historical-reference differences. Evidence is stored before the gate. Requires Databricks execution. No runtime result is claimed by this notebook text.

In [0]:
%sql
SELECT data_quality_attribute, status, action, failed_rows, total_rows, percentage,
       data_quality_check, details
FROM `04-validation`.silver_dq_results
WHERE table_name = 'silver_project_flood_map'
  AND validation_run_id = project_flood_validation_run_id
ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'FLAG' THEN 2 ELSE 3 END,
         data_quality_attribute, data_quality_check;